# Train A Self-Driving Lab Policy on H100

This notebook is designed for Jupyter GPU nodes such as H100 clusters.
It uses the notebook-friendly helpers in `training_script.py` to build prompts from the same self-driving lab environment state used by `run_agent.py`, preview reference actions, and launch GRPO training without shelling out to the CLI.

In [ ]:
%pip install -q -U torch transformers datasets trl accelerate matplotlib huggingface_hub

# Optional extras used by some reward-scoring paths.
%pip install -q -U sentence-transformers gseapy

In [ ]:
from pathlib import Path

import torch

from training_script import build_prompt_examples, make_training_args, run_training

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

Path("artifacts").mkdir(exist_ok=True)

In [ ]:
args = make_training_args(
    model_id="Qwen/Qwen3.5-0.8B",
    output_dir="artifacts/grpo-h100",
    dataset_episodes=32,
    rollout_steps=10,
    collection_policy="heuristic",
    reward_backend="local",
    domain_randomise=True,
    num_generations=4,
    max_completion_length=160,
    max_prompt_length=1280,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1.0,
    logging_steps=1,
    save_steps=25,
    trust_remote_code=True,
    dry_run=False,
    seed=42,
)

args

In [ ]:
preview_examples = build_prompt_examples(
    dataset_episodes=1,
    rollout_steps=args.rollout_steps,
    collection_policy=args.collection_policy,
    scenario_names=["cardiac_disease_de"],
    seed=args.seed,
    domain_randomise=args.domain_randomise,
)

print(preview_examples[0]["prompt"][:3500])
print("\nReference action:\n", preview_examples[0]["reference_action"])


In [ ]:
# Optional smoke test before a full run.
dry_run_args = make_training_args(**{**vars(args), "dry_run": True})
dry_run_result = run_training(dry_run_args)
len(dry_run_result["examples"])

In [ ]:
from IPython.display import Image, display

train_result = run_training(args)
for name, plot_path in train_result["plot_paths"].items():
    print(name, plot_path)
    display(Image(filename=plot_path))